# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UnsoundMouse/flyrankaiw01_research_question/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

## 1. The contract, in plain words

1. **What one row means for my lane:** In the raw table, one row is one
   `report_date × client_hash_id × content_hash_id` — a single page's search performance on a
   single day. For scoring, I roll this up to **one row = one page for one client, aggregated
   over March 2026** — the same grain my Week 1–2 work used, just now built from the real
   warehouse instead of the starter CSV.
2. **Table(s):** `fact_content_daily_performance`, partition `month=2026-03` only. I don't need
   `dim_content` or `dim_clients` for the core contract — see "deliberately excluded" below.
3. **Time window:** March 2026 (`2026-03-01` → `2026-03-31`), a mid-panel month, not the sealed
   `_sample` final month. Decision moment: end of day `2026-03-31` — every feature must be
   computable using only data through that date.
4. **What I'd predict or rank (proxy):** `ctr_gap` = a page's position-tier-typical CTR minus
   its own observed CTR, aggregated over March — the same proxy target from Week 2, now built
   on real daily data instead of a pre-aggregated 90-day snapshot. Positive = underperforming
   its tier. This is still a **proxy**, not an observed outcome: I have no edit-event log here
   either, so I still can't say a flagged page will actually respond to a rewrite.
5. **One thing I deliberately exclude:** `dim_content.last_optimized_date`. Per the schema, this
   field's timestamps only make sense as *editorial history*, but I have no guarantee every
   value in it pre-dates my March decision moment — using it without checking that first risks
   quietly leaking future information into a "current state" feature. I exclude it this week and
   will only bring it back after verifying its dates with a query, not an assumption.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb
import numpy as np
import pandas as pd
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

BASE    = "hf://datasets/FlyRank/internship-warehouse"
FACT_03 = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')"

print("DECISION MOMENT : 2026-03-31")
print("FEATURE WINDOW  : 2026-03-01 -> 2026-03-31  (31 days, mid-panel, not the sealed final month)")

DECISION MOMENT : 2026-03-31
FEATURE WINDOW  : 2026-03-01 -> 2026-03-31  (31 days, mid-panel, not the sealed final month)


## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_sum_position` | **Feature** | Daily GSC totals within March — knowable at the decision moment because they're observed, not derived from anything downstream |
| `report_date` | **Context** | For aggregating/windowing only, never a model input |
| `client_hash_id`, `content_hash_id` | **Context** | Pseudonyms — grouping and joins only (grouped train/test split), never features |
| `gsc_data_available` | **Feature-gate** | Filters which rows are real vs zero-filled; used to build the frame, not fed to a model directly |
| `ctr_gap` (derived: tier-median CTR − page's March CTR) | **Label / proxy** | The thing I'm scoring. Never a feature — a model given `ctr_gap` to predict `ctr_gap` would trivially "solve" it |
| `mar_ctr` (page's own March CTR, used to build `ctr_gap`) | **Label-derived — excluded from features** | Directly used to compute the proxy target; including it is the leak I deliberately demonstrate and then remove in Section 3 |
| `dim_content.last_optimized_date` | **Excluded** | Unverified whether all values pre-date the decision moment — excluded until checked, not assumed safe |

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, availability)

Three queries, in order: grain, row count + date span, availability filtered with `IS TRUE`.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# QUERY 1 of 3 — GRAIN: zero rows back means the grain holds
print("### QUERY 1 — grain probe on (report_date, client_hash_id, content_hash_id)")
result = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_03} GROUP BY 1, 2, 3 HAVING c > 1 LIMIT 5
""").df()
print(result.to_string(index=False) if len(result) else "  (empty — no duplicate keys, grain holds)")

### QUERY 1 — grain probe on (report_date, client_hash_id, content_hash_id)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  (empty — no duplicate keys, grain holds)


In [3]:
# QUERY 2 of 3 — ROW COUNT AND DATE SPAN
print("### QUERY 2 — row count and date span for my March slice")
print(con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT client_hash_id) AS clients,
           COUNT(DISTINCT content_hash_id) AS content_items,
           MIN(report_date) AS first_date, MAX(report_date) AS last_date
    FROM {FACT_03}
""").df().to_string(index=False))

### QUERY 2 — row count and date span for my March slice
 n_rows  clients  content_items first_date  last_date
9841378       55         331437 2026-03-01 2026-03-31


In [4]:
# QUERY 3 of 3 — AVAILABILITY, filtered with IS TRUE (not = TRUE — the flag can be NULL)
print("### QUERY 3 — availability: how many rows survive IS TRUE")
print(con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (gsc_data_available IS TRUE)  AS survives_is_true,
           COUNT(*) FILTER (gsc_data_available IS FALSE) AS is_false,
           COUNT(*) FILTER (gsc_data_available IS NULL)  AS is_null,
           ROUND(100.0 * COUNT(*) FILTER (gsc_data_available IS TRUE) / COUNT(*), 1) AS pct_surviving
    FROM {FACT_03}
""").df().to_string(index=False))

print("\n    Confirming IS FALSE rows are zero-filled, not missing:")
print(con.sql(f"""
    SELECT gsc_data_available, COUNT(*) AS rows, SUM(gsc_impressions) AS total_impressions
    FROM {FACT_03} GROUP BY 1 ORDER BY 1
""").df().to_string(index=False))

### QUERY 3 — availability: how many rows survive IS TRUE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  survives_is_true  is_false  is_null  pct_surviving
    9841378           3611061   6230317        0           36.7

    Confirming IS FALSE rows are zero-filled, not missing:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 gsc_data_available    rows  total_impressions
              False 6230317                0.0
               True 3611061        280657589.0


### The five-feature frame

One row = one `(client_hash_id, content_hash_id)` page, aggregated over March only. Every
feature line states why it's knowable at the `2026-03-31` decision moment.

In [5]:
frame = con.sql(f"""
WITH mar AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions)  AS mar_impr,
         SUM(gsc_clicks)       AS mar_clicks,
         SUM(gsc_sum_position) AS mar_sum_pos,
         COUNT(*)              AS mar_days,
         SUM(gsc_impressions) FILTER (report_date <  DATE '2026-03-16') AS h1_impr,
         SUM(gsc_impressions) FILTER (report_date >= DATE '2026-03-16') AS h2_impr
  FROM {FACT_03}
  WHERE gsc_data_available IS TRUE
  GROUP BY 1, 2
)
SELECT client_hash_id, content_hash_id,
       mar_impr * 1.0 / mar_days                              AS daily_impressions,
       mar_sum_pos * 1.0 / NULLIF(mar_impr, 0)                 AS avg_position,
       mar_clicks * 100.0 / NULLIF(mar_impr, 0)                AS mar_ctr,
       (h2_impr - h1_impr) * 1.0 / NULLIF(h2_impr + h1_impr,0) AS momentum_h2_vs_h1,
       mar_days                                                AS days_with_data
FROM mar
WHERE mar_days >= 15 AND mar_impr >= 30
""").df()

# position tier + tier-median CTR + the proxy target, all computed from March only
def tier(p):
    if p <= 3: return "top_3"
    if p <= 10: return "page_1"
    if p <= 20: return "striking"
    if p <= 50: return "page_3_5"
    return "deep"

frame["position_tier"] = frame["avg_position"].apply(tier)
tier_median = frame.groupby("position_tier")["mar_ctr"].transform("median")
frame["ctr_gap"] = tier_median - frame["mar_ctr"]
frame["underperforms"] = (frame["ctr_gap"] > 0).astype(int)

FEATURES = ["daily_impressions", "avg_position", "momentum_h2_vs_h1", "days_with_data"]
# note: mar_ctr is intentionally NOT in this list -- it's the leak, added deliberately in Section 4

print(f"frame: {frame.shape[0]:,} rows | clients: {frame['client_hash_id'].nunique()}")
print(f"underperforms base rate: {frame['underperforms'].mean()*100:.1f}%")
print()
print(frame[FEATURES].describe().round(3).to_string())

print("""
Feature availability, one line each:
  daily_impressions  -- knowable at decision moment: raw March GSC totals, nothing forward-looking
  avg_position       -- knowable: mean GSC position over March, observed not predicted
  momentum_h2_vs_h1   -- knowable: compares first vs second half of March only, no April data used
  days_with_data      -- knowable: a count of March calendar days, trivially available
  position_tier       -- knowable: bucketed from avg_position, same March-only information
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

frame: 116,539 rows | clients: 40
underperforms base rate: 36.7%

       daily_impressions  avg_position  momentum_h2_vs_h1  days_with_data
count         116539.000    116539.000         114816.000      116539.000
mean              79.038        15.909              0.078          28.012
std              214.509        16.449              0.331           4.506
min                1.214         0.000             -0.999          15.000
25%                5.897         5.077             -0.120          26.000
50%               19.097         8.867              0.067          31.000
75%               69.097        21.173              0.260          31.000
max            21280.138       106.891              1.000          31.000

Feature availability, one line each:
  daily_impressions  -- knowable at decision moment: raw March GSC totals, nothing forward-looking
  avg_position       -- knowable: mean GSC position over March, observed not predicted
  momentum_h2_vs_h1   -- knowable: compares 

### The trap: plant a label-derived column, watch it, delete it

`ctr_gap` (and its binary version `underperforms`) is *computed directly from* `mar_ctr`. If
`mar_ctr` is also handed to a model as a feature, the model doesn't need to learn anything — it
can just invert the formula. This is the same shape of mistake the starter data dictionary
warns about with `trend_direction`/`trend_pct` never being features for `is_declining_label`.

In [6]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

y = frame["underperforms"].to_numpy()
groups = frame["client_hash_id"].to_numpy()

def quick_score(cols, tag):
    X = frame[cols]
    aucs = []
    for tr, te in GroupKFold(n_splits=4).split(X, y, groups):
        m = HistGradientBoostingClassifier(max_iter=120, random_state=0).fit(X.iloc[tr], y[tr])
        p = m.predict_proba(X.iloc[te])[:, 1]
        aucs.append(roc_auc_score(y[te], p))
    print(f"  {tag:40s} ROC-AUC = {np.mean(aucs):.4f}")
    return np.mean(aucs)

print(f"base rate: {y.mean():.3f}  (validation: GroupKFold by client_hash_id, 4 folds)\n")
honest = quick_score(FEATURES, "4 honest features")
leaked = quick_score(FEATURES + ["mar_ctr"], "+ mar_ctr  <-- DELIBERATE LEAK")

print(f"\n  the leak buys {leaked - honest:+.4f} ROC-AUC and means nothing:")
print("  underperforms is DEFINED from mar_ctr via ctr_gap. Handing the model mar_ctr")
print("  lets it read the label instead of predicting it.\n")

# Delete it. The honest number is the one that survives.
print(f"  THE NUMBER I KEEP: ROC-AUC = {honest:.4f} (mar_ctr excluded from features going forward)")

base rate: 0.367  (validation: GroupKFold by client_hash_id, 4 folds)

  4 honest features                        ROC-AUC = 0.8468
  + mar_ctr  <-- DELIBERATE LEAK           ROC-AUC = 1.0000

  the leak buys +0.1532 ROC-AUC and means nothing:
  underperforms is DEFINED from mar_ctr via ctr_gap. Handing the model mar_ctr
  lets it read the label instead of predicting it.

  THE NUMBER I KEEP: ROC-AUC = 0.8468 (mar_ctr excluded from features going forward)


## 4. Data limits

**One named limitation:** filtering to `gsc_data_available IS TRUE` and `mar_days >= 15` and
`mar_impr >= 30` drops an unknown share of pages entirely — low-traffic and newly-onboarded
pages are the most likely to be filtered out, which means this frame is biased toward
already-established, higher-traffic content. A page that's brand new in March, or belongs to a
client with a short GSC history, won't show up here at all — its absence isn't "no
opportunity," it's "not enough data to say." I don't yet know how large this excluded group is
relative to the full March panel — that's a number worth computing before trusting the
opportunity list too far, and I'll check it with a real query once I have this contract locked.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

- [x] Five plain-words contract answers
- [x] Exactly three verification queries with outputs visible (availability checked with `IS TRUE`)
- [x] A five-feature frame with an "available when?" line per feature
- [x] The deliberate-leak experiment shown and removed
- [x] One named limitation of the slice
- [x] No client names, URLs, or private queries anywhere
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.